In [1]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv("data/entities/louvre_entities.csv")

# Nettoyage
df = df.dropna()
df = df[~df["entity"].str.contains("http", case=False)]
df = df[df["entity"].str.len() < 40]
df = df.drop_duplicates(subset=["entity", "source_file"])

# Types utiles seulement
valid_types = ["PERSON", "WORK_OF_ART", "GPE", "ORG"]
df = df[df["entity_type"].isin(valid_types)]

# Normalisation
def normalize_type(t):
    mapping = {
        "GPE": "LOCATION",
        "ORG": "ORGANIZATION"
    }
    return mapping.get(t, t)

df["entity_type"] = df["entity_type"].apply(normalize_type)

print("Nombre d'entités :", len(df))

Nombre d'entités : 2190


In [5]:
top_entities = pd.read_csv("data/entities/top_20_entities.csv")
top_list = set(top_entities["entity"])

df = df[df["entity"].isin(top_list)]

print("Après filtrage :", len(df))

Après filtrage : 44


In [7]:
G = nx.DiGraph()

# Nœuds
for _, row in df.iterrows():
    G.add_node(row["entity"], type=row["entity_type"])

# Relations limitées
grouped = df.groupby("source_file")

for file, group in grouped:

    entities = group[["entity", "entity_type"]].values

    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):

            e1, t1 = entities[i]
            e2, t2 = entities[j]

            if t1 == "WORK_OF_ART" and t2 == "PERSON":
                G.add_edge(e1, e2, label="CREATED_BY")

            elif t2 == "WORK_OF_ART" and t1 == "PERSON":
                G.add_edge(e2, e1, label="CREATED_BY")

            elif t1 == "WORK_OF_ART" and t2 == "LOCATION":
                G.add_edge(e1, e2, label="LOCATED_IN")

            elif t2 == "WORK_OF_ART" and t1 == "LOCATION":
                G.add_edge(e2, e1, label="LOCATED_IN")

print("Nodes:", len(G.nodes))
print("Edges:", len(G.edges))

Nodes: 12
Edges: 0


In [9]:
focus = "Mona Lisa"

if focus in G:
    neighbors = list(G.neighbors(focus))
    sub_nodes = neighbors + [focus]
    G_sub = G.subgraph(sub_nodes)
else:
    G_sub = G

In [11]:
focus = "Mona Lisa"

if focus in G:
    neighbors = list(G.neighbors(focus))
    sub_nodes = neighbors + [focus]
    G_sub = G.subgraph(sub_nodes)
else:
    G_sub = G

In [27]:
net = Network(height="700px", width="100%", directed=True)

for node in G_sub.nodes:
    t = G_sub.nodes[node]["type"]

    if t == "PERSON":
        color = "lightgreen"
    elif t == "WORK_OF_ART":
        color = "skyblue"
    elif t == "LOCATION":
        color = "orange"
    else:
        color = "grey"

    net.add_node(node, label=node, color=color)

for src, tgt, data in G_sub.edges(data=True):
    net.add_edge(src, tgt, label=data["label"])

net.write_html("louvre_graph.html")

In [13]:
texts = []
folder = "data/cleaned/"

for file in os.listdir(folder):
    with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
        texts.append(f.read())

print("Nombre de documents :", len(texts))

Nombre de documents : 4


In [15]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

In [17]:
def retrieve(query, k=2):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, X)

    top_idx = scores[0].argsort()[-k:][::-1]
    return [texts[i] for i in top_idx]

In [19]:
def enrich_with_kg(query):
    related = []

    for node in G.nodes:
        if node.lower() in query.lower():
            neighbors = list(G.neighbors(node))
            related.extend(neighbors)

    return list(set(related))

In [21]:
def answer_question(query):

    docs = retrieve(query)
    kg_info = enrich_with_kg(query)

    context = " ".join(docs)

    # Réponses simples intelligentes
    if "mona lisa" in query.lower() and "who" in query.lower():
        if kg_info:
            return f"The Mona Lisa was created by {kg_info[0]}."

    if "where" in query.lower() and "mona lisa" in query.lower():
        if kg_info:
            return f"The Mona Lisa is located in {kg_info[0]}."

    return f"""
Question: {query}

Context:
{context[:500]}

KG Entities:
{", ".join(kg_info)}

Answer:
Based on the documents and the knowledge graph, the answer is related to the context above.
"""

In [23]:
print(answer_question("Who created the Mona Lisa?"))
print(answer_question("Where is the Mona Lisa located?"))


Question: Who created the Mona Lisa?

Context:
--- title: Mona Lisa - Wikipedia author: Authority control databases url: https://en.wikipedia.org/wiki/Mona_Lisa hostname: wikipedia.org sitename: Wikimedia Foundation, Inc. date: 2002-08-12 --- Mona Lisa | Mona Lisa | | |---|---| | Italian: la Gioconda, Monna Lisa, French: la Joconde | | | Artist | Leonardo da Vinci | | Year | c. 1503–1506, perhaps continuing until c. 1517 | | Medium | Oil on poplar panel | | Subject | Lisa del Giocondo | | Dimensions | 77 cm × 53 cm (30 in × 21 in) | | Locati

KG Entities:


Answer:
Based on the documents and the knowledge graph, the answer is related to the context above.


Question: Where is the Mona Lisa located?

Context:
--- title: Mona Lisa - Wikipedia author: Authority control databases url: https://en.wikipedia.org/wiki/Mona_Lisa hostname: wikipedia.org sitename: Wikimedia Foundation, Inc. date: 2002-08-12 --- Mona Lisa | Mona Lisa | | |---|---| | Italian: la Gioconda, Monna Lisa, French: la Jo